<a href="https://colab.research.google.com/github/AllenGeorge08/NeuralNetworksFromScratch/blob/master/MakemorePart2FromScratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
path = kagglehub.dataset_download("skaspari/makemore-names")
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

100%|██████████| 102k/102k [00:00<00:00, 310kB/s]

Extracting files...


In [51]:
words = open(f'{path}/names.txt','r').read().splitlines()
len(words)

32033

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{0: '.', 1: 'b', 2: 'c', 3: 'd', 4: 'e', 5: 'f', 6: 'g', 7: 'h', 8: 'i', 9: 'j', 10: 'k', 11: 'l', 12: 'm', 13: 'n', 14: 'o', 15: 'p', 16: 'q', 17: 'r', 18: 's', 19: 't', 20: 'u', 21: 'v', 22: 'w', 23: 'x', 24: 'y', 25: 'z'}


In [37]:
# Building the dataset
# Context Length /block of prediction.
def build_dataset(words):
  block_size=3
  X,Y=[],[]

  for w in words:
      context=[0]*block_size  #[0,0,0] -> e (0 = .) bcz numerically [0,0,0]-> 5 i.e then we slide the context ..e or 005
      for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)

        context = context[1:] + [ix]   #[0 0 0 ] + [5] = [0 0 5] 5=e , then m comes
  X = torch.tensor(X)
  Y = torch.tensor(Y)

  return X,Y

In [52]:
# dataset split
import random

# to shuffle the words, also seed makes the shuffling determenistic
random.seed(42)
random.shuffle(words)

# 80% train , 10% dev , 10%test
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,Ytr  =build_dataset(words[:n1])
Xdev,Ydev = build_dataset(words[n1:n2])  #Questions used while tuning the approach
Xte,Yte = build_dataset(words[n2:])

Xte.shape,Yte.shape,Xtr.shape,Ytr.shape,Xdev.shape,Ydev.shape
#

(torch.Size([22866, 3]),
 torch.Size([22866]),
 torch.Size([182625, 3]),
 torch.Size([182625]),
 torch.Size([22655, 3]),
 torch.Size([22655]))

In [43]:
g = torch.Generator().manual_seed(2147483647) #this will create samme random numbers when you run it with that same generator
C  = torch.randn((27,10),generator=g)  #embedding table 27 chracters with 10 numbers each #Q why 10 numbers
W1 = torch.randn((30,200),generator=g)  # 3 characters, 10 embedding num each... so 30 inputss and send them into 200 neurons..
b1 = torch.randn(200,generator=g)  #each output neuron gets one bias(broadcasted)
W2 = torch.randn((200,27),generator=g) #hidden layer has 200 neurons we we need to predict 1/27 characters, hence 200->27
b2 = torch.randn(27,generator=g) #27 chracters -> 27 biases


parameters=[C,W1,b1,W2,b2] #all the numberss model i allowed to learn

for p in parameters:
  p.requires_grad = True

In [73]:

for i in range(10000):
  # Mini-batch #Q
  ix = torch.randint(0,Xtr.shape[0],(32,))

  # Forward pass
  emb = C[Xtr[ix]]
  h = torch.tanh(emb.view(-1,30)@W1+b1)
  logits = h@W2+b2

  #Loss
  loss = F.cross_entropy(logits,Ytr[ix])

  # Backward pass
  for p in parameters: #reset gradients
    p.grad = None

  loss.backward()

  # Update
  lr = 0.001
  for p in parameters:
    p.data += -lr*p.grad

  if i%1000==0:
    print(i,loss.item())

0 2.1471540927886963
1000 2.0182833671569824
2000 1.9999974966049194
3000 2.0545899868011475
4000 2.246446371078491
5000 1.9129538536071777
6000 2.164592742919922
7000 2.1574478149414062
8000 2.2548866271972656
9000 1.7846002578735352


In [74]:
# Evaluating on training data
emb = C[Xtr]
h = torch.tanh(emb.view(-1,30)@W1+b1)
logits = h@W2+b2

# NLL + softmax
loss = F.cross_entropy(logits,Ytr)
print("Train loss: ",loss.item())


Train loss:  2.07747220993042


In [75]:
# Evaluate on dev data
emb = C[Xdev]
h = torch.tanh(emb.view(-1, 30) @ W1 + b1)
logits = h @ W2 + b2

loss = F.cross_entropy(logits, Ydev)

print("dev loss:", loss.item())

dev loss: 2.1127898693084717


In [76]:
emb = C[Xte]

h = torch.tanh(
    emb.view(-1, 30) @ W1 + b1
)

logits = h @ W2 + b2

loss = F.cross_entropy(logits, Yte)

print("test loss:", loss.item())

test loss: 2.1125645637512207


In [63]:
g = torch.Generator().manual_seed(2147483647)

for _ in range(20):

    out = []
    context = [0, 0, 0]

    while True:

        # Convert context to tensor
        emb = C[torch.tensor([context])]

        # Forward pass
        h = torch.tanh(
            emb.view(1, 30) @ W1 + b1
        )

        logits = h @ W2 + b2

        # Convert logits → probabilities
        probs = F.softmax(logits, dim=1)

        # Sample next character
        ix = torch.multinomial(
            probs,
            num_samples=1,
            generator=g
        ).item()

        # Add character
        context = context[1:] + [ix]
        out.append(itos[ix])

        # Stop at '.'
        if ix == 0:
            break

    print(''.join(out))

dey.
n.
mini.
villi.
k.
.
zil.
m.
k.
ni.
obron.
leel.
li.
el.
sh.
jurvik.
cis.
j.
ury.
jer.
